In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
# Load the Kaggle competition files.
# Keeping these paths centralized makes it easy to rerun in Kaggle without touching later cells.
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e5"

df = pd.read_csv(f"{DATA_DIR}/train.csv")
TEST_PATH = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import time


# Drop id
df = df.drop(columns=['id'])

# Prepare categorical columns
cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop(columns=['PitNextLap'])
y = df['PitNextLap']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# Initialize and train XGBoost
print("Training XGBoost baseline...")
start_time = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1
)

clf.fit(X_train, y_train)
elapsed = time.time() - start_time
print(f"Training completed in {elapsed:.2f} seconds.")

# Evaluate
y_train_pred = clf.predict_proba(X_train)[:, 1]
y_val_pred = clf.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"Train ROC-AUC: {train_auc:.5f}")
print(f"Validation ROC-AUC: {val_auc:.5f}")

X_train shape: (351312, 14), X_val shape: (87828, 14)
Training XGBoost baseline...
Training completed in 8.50 seconds.
Train ROC-AUC: 0.97477
Validation ROC-AUC: 0.93986


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)
# Drop id from features but keep for output
train_ids = train_df['id']
test_ids = test_df['id']
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])
# Feature engineering
print("Engineering features...")
for df in [train_df, test_df]:
    df['Estimated_Total_Laps'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df['Estimated_Total_Laps'] = df['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df['Remaining_Laps'] = df['Estimated_Total_Laps'] - df['LapNumber']
    df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-8)
    df['TyreLife_x_Degradation'] = df['TyreLife'] * df['Cumulative_Degradation']
# Median pit tyre life
pit_stops = train_df[train_df['PitNextLap'] == 1]
median_pit = pit_stops.groupby(['Race', 'Compound', 'Stint'])['TyreLife'].median().reset_index()
median_pit.rename(columns={'TyreLife': 'MedianPitTyreLife'}, inplace=True)
global_median = pit_stops['TyreLife'].median()
comp_median = pit_stops.groupby('Compound')['TyreLife'].median().to_dict()
train_df = train_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
train_df['MedianPitTyreLife'] = train_df['MedianPitTyreLife'].fillna(train_df['Compound'].map(comp_median)).fillna(global_median)
train_df['TyreLife_to_MedianPitRatio'] = train_df['TyreLife'] / (train_df['MedianPitTyreLife'] + 1e-8)
test_df = test_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
test_df['MedianPitTyreLife'] = test_df['MedianPitTyreLife'].fillna(test_df['Compound'].map(comp_median)).fillna(global_median)
test_df['TyreLife_to_MedianPitRatio'] = test_df['TyreLife'] / (test_df['MedianPitTyreLife'] + 1e-8)
# Median lap time
median_lap = train_df.groupby(['Race','Compound'])['LapTime (s)'].median().reset_index()
median_lap.rename(columns={'LapTime (s)': 'MedianLapTimeRC'}, inplace=True)
global_lap = train_df['LapTime (s)'].median()
train_df = train_df.merge(median_lap, on=['Race','Compound'], how='left')
train_df['MedianLapTimeRC'] = train_df['MedianLapTimeRC'].fillna(global_lap)
train_df['LapTime_Ratio'] = train_df['LapTime (s)'] / (train_df['MedianLapTimeRC'] + 1e-8)
test_df = test_df.merge(median_lap, on=['Race','Compound'], how='left')
test_df['MedianLapTimeRC'] = test_df['MedianLapTimeRC'].fillna(global_lap)
test_df['LapTime_Ratio'] = test_df['LapTime (s)'] / (test_df['MedianLapTimeRC'] + 1e-8)
# Categorical conversion
cat_cols = ['Driver','Compound','Race']
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
# Prepare features and target
y_train = train_df['PitNextLap']
X_train = train_df.drop(columns=['PitNextLap'])
X_test = test_df.copy()
print('Feature matrix shapes:', X_train.shape, X_test.shape)
# XGBoost model
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    min_child_weight=3,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='auc'
)
print('Training final model on full data...')
start = time.time()
clf.fit(X_train, y_train)
print(f'Training completed in {time.time() - start:.2f} seconds.')
print('Predicting on test set...')
test_pred = clf.predict_proba(X_test)[:,1]
pred_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_pred})


Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Engineering features...
Feature matrix shapes: (439140, 22) (188165, 22)
Training final model on full data...
Training completed in 52.83 seconds.
Predicting on test set...


In [6]:
OUTPUT_PATH = "submission.csv"

test_pred = clf.predict_proba(X_test)[:,1]

pred_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_pred
})

pred_df.to_csv(OUTPUT_PATH, index=False)

print(f'Predictions saved to {OUTPUT_PATH}')

Predictions saved to submission.csv


In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD DATA & INITIAL SETUP
# ==========================================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)

train_ids = train_df['id']
test_ids = test_df['id']
y_train = train_df['PitNextLap']

# Create a unique group identifier for strict validation splitting
# F1 data mein hamesha Race or Year wise split behtareen generalise karta hai
train_df['Race_Group'] = train_df['Year'].astype(str) + "_" + train_df['Race'].astype(str)
groups = train_df['Race_Group']

# ==========================================
# 2. ELITE ENGINE FEATURE PACK
# ==========================================
def engineer_features_v3(df, is_train=True, global_stats=None):
    df_out = df.copy()
    
    # 1. Advanced Structural Features
    df_out['Estimated_Total_Laps'] = (df_out['LapNumber'] / (df_out['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df_out['Estimated_Total_Laps'] = df_out['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df_out['Remaining_Laps'] = df_out['Estimated_Total_Laps'] - df_out['LapNumber']
    
    # 2. Tyre Degradation Physics
    df_out['Degradation_Rate'] = df_out['Cumulative_Degradation'] / (df_out['TyreLife'] + 1e-8)
    df_out['TyreLife_x_Degradation'] = df_out['TyreLife'] * df_out['Cumulative_Degradation']
    df_out['Critical_Degradation_Zone'] = (df_out['Cumulative_Degradation'] > 15).astype(int)
    
    # Strict sorting for time series alignment
    df_out = df_out.sort_values(by=['Year', 'Race', 'Driver', 'LapNumber'])
    
    # 3. Micro-level Time Shifts (Crucial for 0.96+)
    for lag in [1, 2, 3]:
        df_out[f'LapTime_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(lag)
        df_out[f'Position_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['Position'].shift(lag)
        df_out[f'Degradation_Lag_{lag}'] = df_out.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].shift(lag)

    # Momentum Deltas
    df_out['LapTime_Delta_1'] = df_out['LapTime (s)'] - df_out['LapTime_Lag_1']
    df_out['LapTime_Delta_2'] = df_out['LapTime_Lag_1'] - df_out['LapTime_Lag_2']
    df_out['Position_Delta_1'] = df_out['Position'] - df_out['Position_Lag_1']
    
    # Rolling averages over windows
    df_out['LapTime_RollMean_3'] = df_out.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df_out['LapTime_RollStd_3'] = df_out.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0)
    df_out['Degradation_RollMean_3'] = df_out.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    # Anomalous Lap Indicator (Team strategy triggers when pace completely drops)
    df_out['Pace_Drop_Anomaly'] = df_out['LapTime (s)'] - df_out['LapTime_RollMean_3']
    
    # 4. Macro Strategy Statistics (Pit Windows mapping)
    if is_train:
        pit_stops = df_out[df_out['PitNextLap'] == 1]
        med_pit = pit_stops.groupby(['Race', 'Compound'])['TyreLife'].median().reset_index().rename(columns={'TyreLife': 'MedPitLife'})
        global_med_pit = pit_stops['TyreLife'].median()
        global_med_lap = df_out['LapTime (s)'].median()
        global_stats = (med_pit, global_med_pit, global_med_lap)
    else:
        med_pit, global_med_pit, global_med_lap = global_stats
        
    df_out = df_out.merge(med_pit, on=['Race', 'Compound'], how='left')
    df_out['MedPitLife'] = df_out['MedPitLife'].fillna(global_med_pit)
    df_out['TyreLife_Remaining_Window'] = df_out['MedPitLife'] - df_out['TyreLife']
    df_out['TyreLife_Ratio'] = df_out['TyreLife'] / (df_out['MedPitLife'] + 1e-8)
    
    df_out = df_out.fillna(0)
    
    # Convert Categoricals to Dtype Category
    for col in ['Driver', 'Compound', 'Race']:
        df_out[col] = df_out[col].astype('category')
        
    if is_train:
        return df_out, global_stats
    return df_out

print("Building Elite Engine Features...")
X_train_feat, stats = engineer_features_v3(train_df, is_train=True)
X_test_feat = engineer_features_v3(test_df, is_train=False, global_stats=stats)

# Re-align original index sequence safely
X_train_feat = X_train_feat.set_index('id').loc[train_ids].reset_index()
X_test_feat = X_test_feat.set_index('id').loc[test_ids].reset_index()

# Clean internal processing artifacts
X_train = X_train_feat.drop(columns=['id', 'PitNextLap', 'Race_Group'])
X_test = X_test_feat.drop(columns=['id'])

print(f"Dataset Formatted. Train: {X_train.shape}, Test: {X_test.shape}")

# ==========================================
# 3. GROUP K-FOLD BLENDING & MODEL TUNING
# ==========================================
# GroupKFold ensures that the model generalizes across entire un-seen races
gkf = GroupKFold(n_splits=5)

xgb_preds = np.zeros(len(X_test))
lgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))

oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

cat_features_idx = [X_train.columns.get_loc(col) for col in ['Driver', 'Compound', 'Race']]

print("\nExecuting Strategic Group Validation...")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # --- MODEL 1: DEEP LEARNING GRADIENT XGBOOST ---
    model_xgb = xgb.XGBClassifier(
        n_estimators=3000, max_depth=5, learning_rate=0.015, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=8.0, reg_lambda=15.0, min_child_weight=25, random_state=42 + fold,
        enable_categorical=True, tree_method='hist', n_jobs=-1, eval_metric='auc', early_stopping_rounds=100
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    # --- MODEL 2: DENSE LEAF-WISE LIGHTGBM ---
    model_lgb = LGBMClassifier(
        n_estimators=3000, max_depth=6, num_leaves=31, learning_rate=0.015, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=8.0, reg_lambda=15.0, min_child_weight=25, random_state=42 + fold, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # --- MODEL 3: CATEGORICAL ROBUST CATBOOST ---
    X_tr_cb, X_va_cb, X_test_cb = X_tr.copy(), X_va.copy(), X_test.copy()
    for col in ['Driver', 'Compound', 'Race']:
        X_tr_cb[col] = X_tr_cb[col].astype(str)
        X_va_cb[col] = X_va_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)
        
    model_cat = CatBoostClassifier(
        iterations=3000, depth=5, learning_rate=0.02, l2_leaf_reg=15, random_seed=42 + fold,
        eval_metric='AUC', early_stopping_rounds=100, task_type='CPU', thread_count=-1, verbose=False
    )
    model_cat.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), cat_features=cat_features_idx)
    
    # Map predictions
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    oof_cat[val_idx] = model_cat.predict_proba(X_va_cb)[:, 1]
    
    f_xgb = roc_auc_score(y_va, oof_xgb[val_idx])
    f_lgb = roc_auc_score(y_va, oof_lgb[val_idx])
    f_cat = roc_auc_score(y_va, oof_cat[val_idx])
    print(f"Fold {fold+1} -> XGB: {f_xgb:.5f} | LGB: {f_lgb:.5f} | CatBoost: {f_cat:.5f}")
    
    xgb_preds += model_xgb.predict_proba(X_test)[:, 1] / gkf.n_splits
    lgb_preds += model_lgb.predict_proba(X_test)[:, 1] / gkf.n_splits
    cat_preds += model_cat.predict_proba(X_test_cb)[:, 1] / gkf.n_splits

# Print Consolidated Validation Matrix
print("\n========================= MATRIX =========================")
print(f"OOF XGB AUC:      {roc_auc_score(y_train, oof_xgb):.5f}")
print(f"OOF LGB AUC:      {roc_auc_score(y_train, oof_lgb):.5f}")
print(f"OOF CatBoost AUC: {roc_auc_score(y_train, oof_cat):.5f}")

# Optimized Final Prediction Ensembling
final_test_preds = (xgb_preds * 0.40) + (lgb_preds * 0.40) + (cat_preds * 0.20)

# ==========================================
# 4. SUBMISSION EXPORT
# ==========================================
OUTPUT_PATH = "submission.csv"
pred_df = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': final_test_preds
})
pred_df.to_csv(OUTPUT_PATH, index=False)
print(f'\n[PRO] High-Rank Submission Generated successfully: {OUTPUT_PATH}')

Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Building Elite Engine Features...
Dataset Formatted. Train: (439140, 38), Test: (188165, 38)

Executing Strategic Group Validation...
Fold 1 -> XGB: 0.92544 | LGB: 0.92056 | CatBoost: 0.92537
Fold 2 -> XGB: 0.90479 | LGB: 0.90316 | CatBoost: 0.89040
Fold 3 -> XGB: 0.93445 | LGB: 0.93311 | CatBoost: 0.91102
Fold 4 -> XGB: 0.92710 | LGB: 0.92529 | CatBoost: 0.92957
Fold 5 -> XGB: 0.91224 | LGB: 0.90792 | CatBoost: 0.88454

========================= MATRIX =========================
OOF XGB AUC:      0.92156
OOF LGB AUC:      0.91853
OOF CatBoost AUC: 0.89566

[PRO] High-Rank Submission Generated successfully: submission.csv
